# Batch Normalization (BN) — Complete Study Notes

**Topic:** Deep Learning · **Framework:** PyTorch · **Level:** Beginner → Intermediate/Advanced

## Learning objectives
এই নোট শেষে তুমি পারবে:
- Batch Normalization কেন ব্যবহার করা হয় তা ব্যাখ্যা করতে;
- BN-এর mathematical steps ও learnable parameters বুঝতে;
- `BatchNorm1d`, `BatchNorm2d`, `BatchNorm3d` সঠিকভাবে ব্যবহার করতে;
- training mode ও evaluation mode-এর পার্থক্য বুঝতে;
- BN ছাড়া এবং BN-সহ model তুলনা করতে;
- common errors, limitations এবং বিকল্প normalization methods চিনতে।

> **Key idea:** Batch Normalization mini-batch-এর activation-কে normalize করে, তারপর learnable scale (`gamma`) ও shift (`beta`) প্রয়োগ করে। এটি optimization-কে অনেক ক্ষেত্রে সহজ ও stable করতে পারে, কিন্তু সব model-এ accuracy বাড়বেই—এমন নিশ্চয়তা নেই।

## 1. Batch Normalization কী?

**Batch Normalization (BN)** হলো neural network-এর একটি layer/operation, যা mini-batch-এর feature activation-গুলোর mean ও variance ব্যবহার করে normalize করে। এরপর trainable parameter দিয়ে normalized values-এর scale ও shift শেখে।

### কেন ব্যবহার করা হয়?
- training-কে অনেক ক্ষেত্রে দ্রুত বা বেশি stable করতে সাহায্য করে;
- কিছু ক্ষেত্রে learning rate তুলনামূলক বেশি ব্যবহার করা সহজ করে;
- activation-এর scale নিয়ন্ত্রণে সাহায্য করে;
- কিছু regularization effect দিতে পারে, কারণ training-এ batch statistics বদলায়।

**সতর্কতা:** BN নিজে optimizer নয় এবং overfitting-এর নিশ্চিত সমাধানও নয়। ফলাফল architecture, batch size, data ও training setup-এর ওপর নির্ভর করে.

## 2. Mathematical foundation

একটি feature-এর জন্য mini-batch-এ values ধরা যাক \(x_1, x_2, \ldots, x_m\)।

### Step 1 — Batch mean
\[
\mu_B = \frac{1}{m}\sum_{i=1}^{m}x_i
\]

### Step 2 — Batch variance
\[
\sigma_B^2 = \frac{1}{m}\sum_{i=1}^{m}(x_i-\mu_B)^2
\]

### Step 3 — Normalize
\[
\hat{x}_i = \frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}
\]

`epsilon (ε)` একটি ছোট positive number (PyTorch default `1e-5`), যা denominator-কে numerical instability থেকে রক্ষা করে।

### Step 4 — Learnable scale and shift
\[
y_i = \gamma\hat{x}_i + \beta
\]

- **γ (gamma):** trainable scale; সাধারণত শুরুতে 1।
- **β (beta):** trainable shift; সাধারণত শুরুতে 0।
- **ε (epsilon):** numerical stability-এর জন্য constant।
- **m:** যে dimension-গুলোতে statistics গণনা করা হচ্ছে, সেগুলোর sample count।

Normalization-এর পর output সবসময় ঠিক mean 0 ও variance 1 থাকবে না, কারণ γ ও β শেখার মাধ্যমে values-কে নতুনভাবে scale/shift করতে পারে।

## 3. Internal Covariate Shift: সতর্কভাবে বুঝি

BN-কে ঐতিহাসিকভাবে layer input-এর distribution বদল কমানোর সঙ্গে যুক্ত করে ব্যাখ্যা করা হয়েছে। তবে আধুনিক গবেষণায় BN-এর সুবিধাকে শুধু **Internal Covariate Shift** কমানোর মাধ্যমে ব্যাখ্যা করা যথেষ্ট নয়; smoother optimization ও gradient behaviour-ও গুরুত্বপূর্ণ ব্যাখ্যা।

সুতরাং পরীক্ষায় লিখতে পারো: “BN mini-batch statistics ব্যবহার করে activations normalize করে এবং অনেক ক্ষেত্রে optimization-কে আরও stable ও efficient করতে পারে।”

## 4. Batch Normalization কোথায় বসে?

একটি common fully connected block:

`Linear → BatchNorm1d → ReLU`

একটি common CNN block:

`Conv2d → BatchNorm2d → ReLU`

এটি প্রচলিত ordering, বাধ্যতামূলক নিয়ম নয়। Architecture অনুযায়ী ordering বদলাতে পারে। BN সাধারণত convolution/linear transformation-এর পরে এবং activation-এর আগে রাখা হয়।

## 5. PyTorch-এর BatchNorm layers

| Layer | সাধারণ ব্যবহার | Input shape |
|---|---|---|
| `nn.BatchNorm1d(C)` | Fully connected / 1D feature | `(N, C)` অথবা `(N, C, L)` |
| `nn.BatchNorm2d(C)` | 2D CNN feature maps | `(N, C, H, W)` |
| `nn.BatchNorm3d(C)` | 3D volume/video CNN | `(N, C, D, H, W)` |

এখানে `N` = batch size, `C` = channel/feature count, `H/W` = height/width, `D` = depth, `L` = sequence length। `num_features` বলতে channel বা feature dimension বোঝায়—input-এর মোট element সংখ্যা নয়।

In [ ]:
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)

# BatchNorm1d: N samples, C features
x1 = torch.tensor([
    [1.0, 10.0, 100.0],
    [2.0, 20.0, 200.0],
    [3.0, 30.0, 300.0],
    [4.0, 40.0, 400.0],
])
bn1 = nn.BatchNorm1d(num_features=3)
bn1.train()  # explicitly use batch statistics
out1 = bn1(x1)
print("BN1d input shape:", x1.shape)
print("BN1d output shape:", out1.shape)
print("Batch means (approximately 0):", out1.mean(dim=0))
print("Biased batch variances (approximately 1):", out1.var(dim=0, unbiased=False))

In [ ]:
# BatchNorm2d: (N, C, H, W)
x2 = torch.randn(8, 32, 64, 64)
bn2 = nn.BatchNorm2d(num_features=32)
out2 = bn2(x2)
print("BN2d:", x2.shape, "->", out2.shape)

# BatchNorm3d: (N, C, D, H, W)
x3 = torch.randn(4, 16, 8, 32, 32)
bn3 = nn.BatchNorm3d(num_features=16)
out3 = bn3(x3)
print("BN3d:", x3.shape, "->", out3.shape)

## 6. Training mode বনাম Evaluation mode

BN-এর সবচেয়ে গুরুত্বপূর্ণ বিষয়গুলোর একটি হলো mode।

### Training mode: `model.train()`
- বর্তমান mini-batch-এর mean ও variance ব্যবহার করে output normalize করে;
- `running_mean` ও `running_var` update করে;
- model-এর trainable parameters gradient দিয়ে update হতে পারে (যদি backward/optimizer চালানো হয়)।

### Evaluation mode: `model.eval()`
- সাধারণত training-এ জমা হওয়া `running_mean` ও `running_var` ব্যবহার করে;
- running statistics update করে না;
- `eval()` নিজে gradient বন্ধ করে না; inference-এর জন্য সাধারণত `torch.no_grad()` বা `torch.inference_mode()`-ও ব্যবহার করা হয়।

**মনে রাখবে:** `model.eval()` এবং `torch.no_grad()` এক জিনিস নয়। প্রথমটি layer-এর behaviour বদলায়; দ্বিতীয়টি gradient tracking বন্ধ করে।

In [ ]:
bn = nn.BatchNorm1d(3)
x = torch.randn(8, 3)

bn.train()
y_train = bn(x)
print("Training running mean:", bn.running_mean)
print("Training running variance:", bn.running_var)

bn.eval()
with torch.no_grad():
    y_eval = bn(x)
print("Evaluation output shape:", y_eval.shape)
print("Running statistics are used in eval mode by default.")

## 7. `momentum`, `eps`, `affine`, `track_running_stats`

```python
nn.BatchNorm1d(
    num_features=64,
    eps=1e-5,
    momentum=0.1,
    affine=True,
    track_running_stats=True,
)
```

- **`eps`:** denominator-এ যোগ করা ছোট constant।
- **`momentum`:** running statistics update-এর smoothing factor। PyTorch BN-এ update-এর ধারণা: `new_running = (1 - momentum) * old_running + momentum * batch_stat`। এটি optimizer-এর momentum-এর সঙ্গে এক নয়।
- **`affine=True`:** trainable γ ও β ব্যবহার করে।
- **`track_running_stats=True`:** running mean/variance রাখে; evaluation-এ এগুলো ব্যবহার হয়। False হলে সাধারণত evaluation-এও batch statistics ব্যবহার হয়।

**Practical note:** batch খুব ছোট হলে batch statistics noisy হতে পারে। বিশেষত training-এ প্রতি channel-এ মাত্র একটি value থাকলে BN error দিতে পারে।

## 8. Batch Normalization from scratch

নিচের functionটি শিক্ষামূলক উদাহরণ: এটি `dim=0` বরাবর feature-wise mean ও variance ব্যবহার করে এবং normalized values ফেরত দেয়। এটি PyTorch-এর সম্পূর্ণ `BatchNorm1d` replacement নয়, কারণ এখানে trainable γ/β, running statistics, train/eval modes ও state handling নেই.

In [ ]:
def batch_norm_basic(x, eps=1e-5):
    """Educational feature-wise normalization for a 2D tensor (N, C)."""
    if x.ndim != 2:
        raise ValueError("Expected a 2D tensor with shape (N, C)")
    mean = x.mean(dim=0)
    var = x.var(dim=0, unbiased=False)
    x_hat = (x - mean) / torch.sqrt(var + eps)
    return x_hat

x = torch.tensor([
    [2.0, 10.0],
    [4.0, 20.0],
    [6.0, 30.0],
    [8.0, 40.0],
])
print(batch_norm_basic(x))

## 9. Build a neural network with BN

নিচের example-টি regression-style output দেখানোর জন্য। Hidden layers-এ BN আছে, কিন্তু final output layer-এ BN/activation রাখা হয়নি—কারণ output-এর scale target-এর সঙ্গে মিলতে পারে।

In [ ]:
class MLPWithBN(nn.Module):
    def __init__(self, input_dim=10, output_dim=1):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, output_dim),
        )

    def forward(self, x):
        return self.network(x)

model = MLPWithBN(input_dim=10, output_dim=1)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 10. Complete training and evaluation example

এটি synthetic data দিয়ে runnable demonstration; এটি কোনো বাস্তব dataset-এর benchmark নয়। বাস্তব task-এ train/validation/test split এবং appropriate metrics ব্যবহার করতে হবে.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Reproducibility for this demonstration
torch.manual_seed(42)

X = torch.randn(512, 10)
# Synthetic target with a learnable relationship plus small noise
y = (X[:, :3].sum(dim=1, keepdim=True) + 0.1 * torch.randn(512, 1))

dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

model = MLPWithBN(input_dim=10, output_dim=1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        predictions = model(xb)
        loss = criterion(predictions, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    print(f"Epoch {epoch+1}: train MSE = {total_loss / len(dataset):.4f}")

In [ ]:
# Evaluation/inference pattern
model.eval()
with torch.no_grad():
    test_data = torch.randn(5, 10)
    predictions = model(test_data)
print("Predictions shape:", predictions.shape)
print(predictions)

## 11. BatchNorm ছাড়াই বনাম BatchNorm-সহ model

BN performance বাড়ায় কি না তা অনুমান না করে experiment করে দেখতে হবে। Fair comparison-এর জন্য একই dataset split, architecture capacity যতটা সম্ভব সমান, optimizer, learning rate, epoch count ও random seeds ব্যবহার করো। একাধিক seed-এ ফলাফল তুলনা করা আরও ভালো।

তুলনা করার metric task অনুযায়ী বেছে নাও:
- Classification: accuracy, precision, recall, F1, AUROC;
- Regression: MAE, RMSE, R²;
- Training behaviour: train/validation loss curves, convergence speed, stability.

শুধু training loss কম হলেই model ভালো বলা যায় না; validation/test performance ও generalization দেখতে হবে।

## 12. Advantages and limitations

### Advantages
- অনেক architecture-এ optimization সহজ করতে পারে;
- activations-এর scale নিয়ন্ত্রণে সাহায্য করে;
- কিছু ক্ষেত্রে training দ্রুত ও stable হয়;
- trainable scale/shift-এর মাধ্যমে representation flexibility থাকে।

### Limitations
- ছোট batch size-এ statistics noisy হতে পারে;
- train/eval mode ভুল হলে inference ফল বদলে যেতে পারে;
- running statistics train/test distribution mismatch-এ সমস্যা করতে পারে;
- কিছু sequence, tiny-batch, বা particular model setting-এ BN সেরা পছন্দ নয়;
- extra computation এবং running-stat state যোগ করে।

### Common mistakes checklist
- `BatchNorm1d`-এ feature count ভুল দেওয়া;
- CNN input-কে `(N, H, W, C)` format-এ দেওয়া, যখন PyTorch `BatchNorm2d` expects `(N, C, H, W)`;
- inference-এর আগে `model.eval()` না ডাকা;
- validation/test-এর সময় model-কে train mode-এ রেখে running statistics update করা;
- `eval()`-কেই gradient বন্ধ করে মনে করা;
- tiny batch/last incomplete batch-এর কারণে BN failure হওয়া। `drop_last=True` কখনও সাহায্য করতে পারে, কিন্তু এটি সব সমস্যার universal fix নয়।

## 13. BN বনাম অন্য normalization methods

| Method | Statistics সাধারণত কোন axis-এ? | Typical use |
|---|---|---|
| **BatchNorm** | Batch dimension জুড়ে, প্রতিটি feature/channel-এর জন্য | CNN, MLP; যথেষ্ট batch size থাকলে |
| **LayerNorm** | প্রতিটি sample-এর feature dimension জুড়ে | Transformers, sequence models |
| **GroupNorm** | প্রতিটি sample-এর channel-গুলোকে group করে | CNN, ছোট batch size |
| **InstanceNorm** | প্রতিটি sample/channel-এর spatial dimensions জুড়ে | কিছু image generation/style-transfer tasks |

এগুলো একে অন্যের সরাসরি replacement সবসময় নয়। Architecture ও task অনুযায়ী নির্বাচন করতে হয়।

## 14. Batch Normalization in CNNs and ResNet

CNN-এ convolutional output-এর shape সাধারণত `(N, C, H, W)`। `BatchNorm2d(C)` প্রতিটি channel-এর statistics batch ও spatial positions জুড়ে গণনা করে। তাই `C` অবশ্যই convolution-এর output channel সংখ্যার সমান হতে হবে।

ResNet-এর মতো architecture-এ BN বহু convolution block-এর অংশ। তবে pre-activation ResNet-সহ বিভিন্ন architecture-এ ordering আলাদা হতে পারে—model definition অনুসরণ করো।

In [ ]:
# A small CNN block example
class ConvBNReLU(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

cnn_block = ConvBNReLU(3, 16)
image_batch = torch.randn(8, 3, 64, 64)
feature_maps = cnn_block(image_batch)
print("Input:", image_batch.shape)
print("Output:", feature_maps.shape)

## 15. Quick revision / interview questions

1. Batch Normalization কী এবং কেন ব্যবহার করা হয়?
2. BN-এর চারটি প্রধান mathematical step কী?
3. `gamma` ও `beta` কেন trainable?
4. `eps` কী কাজে লাগে?
5. `running_mean` ও `running_var` কী?
6. `model.train()` ও `model.eval()`-এর পার্থক্য কী?
7. `torch.no_grad()` কেন `model.eval()`-এর বিকল্প নয়?
8. PyTorch-এ `BatchNorm2d`-এর input shape কী?
9. Batch size খুব ছোট হলে কী সমস্যা হতে পারে?
10. BatchNorm ও LayerNorm-এর প্রধান পার্থক্য কী?
11. BN সবসময় model accuracy বাড়ায় কি? কেন?
12. BN ছাড়া ও BN-সহ model-এর তুলনা কীভাবে fair করা যায়?

### One-minute summary
Batch Normalization mini-batch statistics দিয়ে activations normalize করে, তারপর learnable `gamma` ও `beta` দিয়ে scale ও shift করে। PyTorch-এ `BatchNorm1d/2d/3d` input-এর feature/channel layout অনুযায়ী ব্যবহার করতে হয়। Training-এ batch statistics ও running statistics update হয়; evaluation-এ defaultভাবে running statistics ব্যবহৃত হয়। BN অনেক ক্ষেত্রে training উন্নত করে, কিন্তু batch size, architecture ও data distribution বিবেচনা করে ব্যবহার করতে হয়.